In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hwk2-task3-aqi.ipynb")

# Task 3: Visualizing AQI during the 2017 Thomas Fire in Santa Barbara County

This task covers topics on the lesson on [time series](https://meds-eds-220.github.io/MEDS-eds-220-course/book/chapters/lesson-7-time-series.html). 

## Instructions 

- First, update the following cell to have a link to *your* Homework 2 GitHub repository:

**UPDATE THIS LINK**
https://github.com/emaliap/eds220-hwk2.git


- Review the [complete rubric for this task](https://docs.google.com/document/d/1x0BoU6IH4cnOR1-n7i9CYQ9wUC37yDpYlQ4j6rCfcsU/edit?tab=t.0) before starting.

- **Make at least 5 commits. Generally, commit every time you finish a major step.** We'll check your repository and view the commit history.

- **Add comments for all your code.** Err on the side of commenting too much for now. Comments should follow best practices.

- **Do not update the top cell with the `otter` import**, this is used internally for grading.


## About the data

In this task you will use [Air Quality Index (AQI)](https://www.airnow.gov/aqi/aqi-basics/) data from the [US Environmental Protection Agency](https://www.epa.gov) to visualize the impact on the AQI of the 2017 [Thomas Fire](https://en.wikipedia.org/wiki/Thomas_Fire) in Santa Barbara County. 

## FINAL CODE

You will use the next code cell to complete the last exercise in the task. Leave it blank for now. 

In [2]:
# Import packages
import pandas as pd

# Read in data
aqi_17 = pd.read_csv('data/daily_aqi_by_county_2017.zip', compression='zip')
aqi_18 = pd.read_csv('data/daily_aqi_by_county_2018.zip', compression='zip')

# Glue the two dataframes together
aqi = pd.concat([aqi_17, aqi_18])

# Simplify column names
aqi.columns = (aqi.columns
                  .str.lower()
                  .str.replace(' ','_')
              )

# Filter to Santa Barbara county
aqi_sb = aqi[aqi['county_name'] == 'Santa Barbara']

# Filter for columns to keep
aqi_sb = aqi_sb[['date', 'aqi', 'category', 'defining_parameter', 'defining_site', 'number_of_sites_reporting']]

# Convert the 'date' column to a datetime object
aqi_sb['date'] = pd.to_datetime(aqi_sb['date'])

# Set the index to be the 'date' column
aqi_sb = aqi_sb.set_index('date')

# Calculate AQI rolling average over 5 days
rolling_average = aqi_sb['aqi'].rolling(window = '5D').mean()

# Add a column for the rolling averages
aqi_sb['five_day_average'] = aqi_sb['aqi'].rolling(window = '5D').mean()

# Plot AQI levels in santa barbara, including the daily measurement and the five day average
aqi_sb[['aqi','five_day_average']].plot(kind='line',
                                        title='AQI levels in Santa Barbara',
                                        xlabel='Date',
                                        ylabel='AQI'
                                        )

FileNotFoundError: [Errno 2] No such file or directory: 'data/daily_aqi_by_county_2017.zip'

## 1

a. Read the [Air Quality Index (AQI) Basics](https://www.airnow.gov/aqi/aqi-basics/) from the AirNow.gov portal.

b. Go to [EPA's website on Air Quality Data Collected at Outdoor Monitors Across the US](https://www.epa.gov/outdoor-air-quality-data). 

c. Under "Download Data", click on "Pre-generated Data Files".

d. Read the "About the data" section.

## 2
a. Back in the "Pre-generated Data Files" site, click on "Tables of Daily AQI".

b. Download the 2017 Daily AQI **by County** ZIP file `daily_aqi_by_county_2017.zip` and the 2018 Daily AQI **by County** ZIP file `daily_aqi_by_county_2018.zip`. Save both files in the same folder as this notebook.

We will be reading the data directly from the local ZIP files. Each ZIP file contains a single CSV that has been compressed to save space.

<!-- BEGIN QUESTION -->

c. In the next code cell read in the data from the local ZIP file using the [`pd.read_csv`] function with the `compression='zip'` parameter added and store it as `aqi_17`.

d. In the same cell, read in the data for the 2018 Daily AQI by County ZIP file and store it as `aqi_18`.

In [ ]:
import pandas as pd

# Read in data
aqi_17 = pd.read_csv('data/daily_aqi_by_county_2017.zip', compression='zip')
aqi_18 = pd.read_csv('data/daily_aqi_by_county_2018.zip', compression='zip')

<!-- END QUESTION -->

## 3
a. and b. Use the next two code cells to look at the head of both data frames. Store your results in `aqi_17_head` and `aqi_18_head`. 

In [ ]:
# Look at the first 5 rows of the aqi_17 dataframe
aqi_17_head = aqi_17.head()
aqi_17_head

In [ ]:
# Look at the first 5 rows of the aqi_18 dataframe
aqi_18_head = aqi_18.head()
aqi_18_head

In [ ]:
grader.check("q3_a")

<!-- BEGIN QUESTION -->

c. Use this code cell to make some other preliminary data exploration of your choosing.

In [ ]:
# How many rows and columns are in the dataframe
print(aqi_17.shape)
print(aqi_18.shape)

# How many non-null values are in each column
print(aqi_17.info())
print(aqi_18.info())

# How many unique values are in each column
print(aqi_17.nunique())
print(aqi_18.nunique())

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

d. Use this markdown cell to explain why you decided to do the exploration in c. and what information you obtained from doing it.

- The number of rows and columns helps me see the overall size of the dataframe. In aqi_17 there are 326801 rows and 10 columns. In aqi_18 there are 327541 rows and 10 columns.

- `info()` helps me assess the amount of missing data and datatype in each column. There are no columns with missing data is from either dataframe. State and county code columns are integers, but should be strings. The date column is an string, not a datetime object.

- The number of unique values shows me how many categories might be in each column or if a column has many unique values. These datasets includes 54 states, 796 or 793 county names, 160 county codes, 365 dates, 341 or 300 AQI values, 6 categories, 5 defining parameters, 2133 or 2105 defining sites, and 35 sites reportings.

<!-- END QUESTION -->

## 4
We currently have two separate data frames. For this exercise we will need to "glue" them one on top of the other. The `pandas` function `pd.concat()` can achieve this. 

Pass `[aqi_17, aqi_18]` as the input of `pd.concat()` and store the output as `aqi`.  
In the next line run `aqi`.

NOTE: When we concatenate data frames like this, without any extra parameters for `pd.concat()` the indices for the two data frames are just "glued together", the index of the resulting data frame is not updated to start from 0. Notice the mismatch between the index of `aqi` and the number of rows in the complete data frame.


In [ ]:
# Glue the two dataframes together
aqi = pd.concat([aqi_17, aqi_18])
aqi

In [ ]:
grader.check("q4")

<!-- BEGIN QUESTION -->

## 5

Run the following code cell and use the next markdown cell to give a line by line explanation of the code below the comment "#Simplify column names". You might need to look up the `pandas` documentation.

In [ ]:
# Initial column names: notice caps and spaces (difficult to work with!)
print(aqi.columns, '\n')

# Simplify column names
aqi.columns = (aqi.columns
                  .str.lower()
                  .str.replace(' ','_')
                )
print(aqi.columns, '\n')

```{python}
aqi.columns = (aqi.columns              # Re-asign the edited columns from the aqi dataframe in place
                  .str.lower()          # Change column names to lower case letters
                  .str.replace(' ','_') # Replace all spaces in the column names with underscores
                )
print(aqi.columns, '\n')                # Print the new column names
```

<!-- END QUESTION -->

## 6
In the next code cell:

a. Select only data from `Santa Barbara` county and store it in a new variable `aqi_sb`.

b. Remove the `state_name`, `county_name`, `state_code` and `county_code` columns from `aqi_sb`.

Your data frame should have the following columns in this order: `date`, `aqi`, `category`, `defining_parameter`, `defining_site`, `number_of_sites_reporting`. 

In [ ]:
aqi.columns

In [ ]:
# Filter to Santa Barbara county
aqi_sb = aqi[aqi['county_name'] == 'Santa Barbara']

# Filter to columns that we want to keep
aqi_sb = aqi_sb[['date', 'aqi', 'category', 'defining_parameter', 'defining_site', 'number_of_sites_reporting']]
aqi_sb

In [ ]:
grader.check("q6a")

 
c. What is the data type of the `date` column? Write your answer in the markdown cell below. 

*Your answer here.*

The 'date' column is a string object data type.

<!-- BEGIN QUESTION -->

## 7
In the following cell:
1. Update the `date` column of `aqi_sb` to have the `datetime64` data type.
2. Update the index of `aqi_sb` to be the `date` column.

In [ ]:
# Convert the 'date' column to a datetime object
aqi_sb['date'] = pd.to_datetime(aqi_sb['date'])

# Set the index to be the 'date' column
aqi_sb = aqi_sb.set_index('date')
aqi_sb

<!-- END QUESTION -->

## 8
In the next cell we will calculate an average over a [rolling window](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.rolling.html) using the `rolling()` method for `pandas.Series`:

- `rolling()` is a lazy method, so we need to specify what we want to calculate over each window before it does something. 
- in this example we use the aggregator function `mean()` to calculate the average over each window
- the parameter '5D' indicates we want the window for our rolling average to be 5 days. 
- we get a `pandas.Series` as output

Store your answer in the `rolling_average` variable. Your result should be a `pandas.Series` indexed by `date`, with the 5-day averages as its values. 

In [ ]:
# Calculate AQI rolling average over 5 days
rolling_average = aqi_sb['aqi'].rolling(window = '5D').mean()
rolling_average

In [ ]:
grader.check("q8")

## 9 

Without creating any new variables, add the mean of the AQI over a 5-day rolling window as a new column named `five_day_average` to the `aqi_sb` data frame. 

In [ ]:
# Add a column for the rolling averages
aqi_sb['five_day_average'] = aqi_sb['aqi'].rolling(window = '5D').mean()

In [ ]:
grader.check("q9")

<!-- BEGIN QUESTION -->

## 10
Make a line plot showing both the daily AQI and the 5-day average (5-day average on top of the AQI). Make necessary updates for the plot to be accurate, informative, and polished, even if simple. You're welcome to use plotting methods and packages beyond what we have covered in class! 

Can you see the AQI going up during the Thomas Fire in December 2017?

In [ ]:
# Plot AQI levels in santa barbara, including the daily measurement and the five day average
aqi_sb[['aqi','five_day_average']].plot(kind='line',
                                        title='AQI levels in Santa Barbara',
                                        xlabel='Date',
                                        ylabel='AQI'
                                        )

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

<!-- END QUESTION -->

## 11


Collect all the relevant code into the first blank cell of the notebook titled "COMPLETE WORKFLOW". This single cell will have the end-to-end workflow: from importing libraries and loading the data, to producing the graph. The *only* output of this cell should be the graph you produced in the previous exercise. Further guidance on what to include in this final workflow is in the [assignment rubric](https://docs.google.com/document/d/1x0BoU6IH4cnOR1-n7i9CYQ9wUC37yDpYlQ4j6rCfcsU/edit?tab=t.0).